# 🛫 Volz NLP Analytics — 
**Google Play Store · 527 Reviews · Aug 2024 – Mar 2026**

| Phase | Description |
|-------|-------------|
| **1** | Data Collection |
| **2** | Preprocessing & Feature Engineering |
| **3** | Exploratory NLP Visualizations |
| **4** | Business Insights & Recommendations |
| **5** | ✨ Enhancement 1 — Arabic NLP (CAMeL Tools) |
| **6** | ✨ Enhancement 2 — App Version Sentiment Tracker |
| **7** | ✨ Enhancement 3 — Payment Method Granularity |
| **8** | ✨ Enhancement 4 — B2B / Corporate Signal Detector |
| **9** | ✨ Enhancement 5 — Diaspora vs Local Segmentation |
| **10** | ✨ Enhancement 6 — Automated Anomaly Alert System |




---
# 🕷️ Phase 1 — Data Collection

Scraping Google Play reviews for the **Volz** app across 5 language/country configurations. Reviews are deduplicated by `reviewId` and saved to CSV.

In [23]:
!pip install google-play-scraper pandas --quiet

In [ ]:
from google_play_scraper import Sort, reviews_all
import pandas as pd
import time

app_id = 'app.volz'

target_configs = [
    {'lang': 'ar', 'country': 'dz'},  # Arabic  — Algeria (local majority)
    {'lang': 'fr', 'country': 'dz'},  # French  — Algeria (educated segment)
    {'lang': 'en', 'country': 'dz'},  # English — Algeria
    {'lang': 'fr', 'country': 'fr'},  # French  — France  (diaspora)
    {'lang': 'en', 'country': 'us'},  # English — USA     (diaspora)
]

all_reviews = []

for config in target_configs:
    print(f"Fetching reviews for {config['lang']}-{config['country']}...")
    try:
        rvs = reviews_all(
            app_id,
            sleep_milliseconds=1000,
            lang=config['lang'],
            country=config['country'],
            sort=Sort.NEWEST
        )
        for r in rvs:
            r['source_lang']    = config['lang']
            r['source_country'] = config['country']
        all_reviews.extend(rvs)
        print(f"  -> Found {len(rvs)} reviews.")
    except Exception as e:
        print(f"  -> Could not fetch for {config}: {e}")

df = pd.DataFrame(all_reviews)

if not df.empty:
    df.drop_duplicates(subset=['reviewId'], inplace=True)
    df.to_csv('volz_total_reviews.csv', index=False, encoding='utf-8-sig')
    print(f"\nSuccess! Total unique reviews collected: {len(df)}")
else:
    print("No reviews were found. Check the App ID or your internet connection.")

---
# 🧹 Phase 2 — Preprocessing & Feature Engineering

Cleans raw text, extracts time features, computes word count, and assigns sentiment labels from star ratings.

| Score | Sentiment |
|-------|-----------|
| ≥ 4   | ✅ Positive |
| 3     | 😐 Neutral  |
| ≤ 2   | ❌ Negative |



In [ ]:
import pandas as pd
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# ── 1. LOAD DATA ──────────────────────────────────────────────────────────────
df = pd.read_csv('volz_total_reviews.csv')

# ── 2. TEXT CLEANING ──────────────────────────────────────────────────────────
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+|\@\w+|\#', '', text, flags=re.MULTILINE)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = " ".join(text.split())
    return text

# ── 3. TIME FEATURES ──────────────────────────────────────────────────────────
df['at'] = pd.to_datetime(df['at'], errors='coerce')
df['year_month'] = df['at'].dt.to_period('M').astype(str)
df['day_of_week'] = pd.Categorical(
    df['at'].dt.day_name(),
    categories=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'],
    ordered=True
)
df['hour'] = df['at'].dt.hour

# ── 4. TEXT FEATURES ──────────────────────────────────────────────────────────
df['clean_content'] = df['content'].apply(clean_text)
df['word_count']    = df['content'].apply(lambda x: len(str(x).split()))

# ── 5. SENTIMENT LABELS ───────────────────────────────────────────────────────
def get_sentiment(score):
    if score >= 4:  return 'Positive'
    elif score == 3: return 'Neutral'
    else:            return 'Negative'

df['sentiment'] = df['score'].apply(get_sentiment)

# ── ZEPHYR THEME COLORS ──────────────────────────────────────────────────────
ZEPHYR = {
    "primary": "#3459E6",
    "success": "#2FB380",
    "warning": "#F39C12",
    "danger": "#DA292E",
    "info": "#17A2B8",
    "light": "#F8F9FA",
    "grid": "#E9ECEF",
    "text": "#212529"
}

# Apply Zephyr style
sns.set_theme(style="whitegrid")

plt.rcParams.update({
    "axes.facecolor": ZEPHYR["light"],
    "figure.facecolor": "white",
    "axes.edgecolor": ZEPHYR["grid"],
    "axes.labelcolor": ZEPHYR["text"],
    "text.color": ZEPHYR["text"],
    "xtick.color": ZEPHYR["text"],
    "ytick.color": ZEPHYR["text"],
    "grid.color": ZEPHYR["grid"]
})


# ── 6. SAVE ───────────────────────────────────────────────────────────────────
df.to_csv('volz_preprocessed_reviews.csv', index=False)
print("Preprocessing complete. Saved to 'volz_preprocessed_reviews.csv'")


# ── 7. VISUALIZATIONS ─────────────────────────────────────────────────────────

# Figure 1: Score distribution + Sentiment pie
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Zephyr score palette
score_colors = {
    1: ZEPHYR["danger"],
    2: "#E67E22",
    3: ZEPHYR["warning"],
    4: ZEPHYR["info"],
    5: ZEPHYR["success"]
}

df_score = df.copy()
df_score['score_str'] = df_score['score'].astype(str)

sns.countplot(
    data=df_score,
    x='score',
    hue='score',
    palette=score_colors,
    ax=axes[0],
    legend=False
)

axes[0].set_title(
    'Distribution of Scores (1–5)',
    fontsize=13,
    fontweight='bold',
    color=ZEPHYR["primary"]
)

axes[0].spines[['top','right']].set_visible(False)


# Sentiment pie
sent_counts = df['sentiment'].value_counts()

sent_order  = ['Positive', 'Neutral', 'Negative']
sent_colors = [
    ZEPHYR["success"],
    ZEPHYR["warning"],
    ZEPHYR["danger"]
]

ordered_counts = [sent_counts.get(s, 0) for s in sent_order]

axes[1].pie(
    ordered_counts,
    labels=sent_order,
    autopct='%1.1f%%',
    colors=sent_colors,
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)

axes[1].set_title(
    'Overall Sentiment Split',
    fontsize=13,
    fontweight='bold',
    color=ZEPHYR["primary"]
)

plt.tight_layout()
plt.savefig('sentiment_overview.png', dpi=150, bbox_inches='tight')
plt.show()



# ── Figure 2: Monthly trends ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

monthly_vol = df.groupby('year_month').size()

monthly_vol.plot(
    kind='line',
    marker='o',
    color=ZEPHYR["primary"],
    linewidth=2,
    ax=axes[0]
)

axes[0].set_title(
    'Monthly Review Volume',
    fontsize=13,
    fontweight='bold',
    color=ZEPHYR["primary"]
)

axes[0].spines[['top','right']].set_visible(False)
axes[0].set_xlabel('')


monthly_rating = df.groupby('year_month')['score'].mean()

monthly_rating.plot(
    kind='line',
    marker='s',
    color=ZEPHYR["warning"],
    linewidth=2,
    ax=axes[1]
)

axes[1].set_title(
    'Monthly Average Rating Trend',
    fontsize=13,
    fontweight='bold',
    color=ZEPHYR["primary"]
)

axes[1].set_ylim(1, 5)

axes[1].axhline(
    y=3.0,
    color=ZEPHYR["danger"],
    linestyle='--',
    alpha=0.7,
    label='Threshold 3.0'
)

axes[1].legend()

axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('trends_over_time.png', dpi=150, bbox_inches='tight')
plt.show()



# ── Figure 3: Activity heatmap ────────────────────────────────────────────────
plt.figure(figsize=(14, 6))

heatmap_data = df.groupby(['day_of_week', 'hour']).size().unstack(fill_value=0)

sns.heatmap(
    heatmap_data,
    cmap='Blues',
    linewidths=0.3,
    linecolor='white'
)

plt.title(
    'User Activity Heatmap (Day vs Hour)',
    fontsize=13,
    fontweight='bold',
    color=ZEPHYR["primary"]
)

plt.tight_layout()
plt.savefig('activity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()



# ── Figure 4: Top Keywords ────────────────────────────────────────────────────

BASIC_STOPS = {'the','and','to','is','it','for','app','volz','this','you','with','that','are'}

def get_top_keywords(text_series, n=15):
    words = []
    for text in text_series.dropna():
        tokens = re.findall(r'\b[a-z]{4,}\b', str(text).lower())
        words.extend([w for w in tokens if w not in BASIC_STOPS])
    return Counter(words).most_common(n)


top_kw = pd.DataFrame(
    get_top_keywords(df['clean_content']),
    columns=['Word','Count']
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_kw,
    x='Count',
    y='Word',
    palette='Blues_r',
    hue='Word',
    legend=False
)

plt.title(
    'Top 15 Most Frequent Keywords',
    fontsize=13,
    fontweight='bold',
    color=ZEPHYR["primary"]
)

plt.tight_layout()
plt.savefig('top_keywords.png', dpi=150, bbox_inches='tight')
plt.show()
print("All Phase 2 visualizations saved.")

---
# 📊 Phase 3 — Exploratory NLP Visualizations

General overview across **all reviews**: sentiment split, word clouds, n-grams, source breakdown, review length, and sentiment over time.


In [ ]:
!pip install wordcloud scikit-learn networkx nltk camel-tools --quiet
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("All NLP libraries ready.")

### 🌥️ Figure 1 — Word Clouds by Sentiment
Frequency maps of words per sentiment group. Compare vocabulary between happy and unhappy users.

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS
from nltk.corpus import stopwords

df = pd.read_csv('volz_preprocessed_reviews.csv')

LATIN_STOPWORDS = set(STOPWORDS) | set(stopwords.words('english')) | set(stopwords.words('french')) | {
    'app', 'volz', 'application', 'this', 'that', 'its', 'very',
    'mais', 'pour', 'avec', 'dans', 'une', 'des', 'les', 'est',
    'pas', 'plus', 'tout', 'bien', 'aussi', 'avoir', 'cest', 'quil',
    'really', 'just', 'even', 'also', 'would', 'could', 'should'
}

sentiments = ['Positive', 'Neutral', 'Negative']
cmaps      = ['YlGn',     'YlOrBr',   'Reds'   ]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#0A2342')
fig.suptitle('Word Clouds by Sentiment', fontsize=20, fontweight='bold', color='white', y=1.02)

for ax, sentiment, cmap in zip(axes, sentiments, cmaps):
    subset   = df[df['sentiment'] == sentiment]['clean_content'].dropna()
    text     = ' '.join(subset)
    words    = re.findall(r'\b[a-z]{4,}\b', text.lower())
    filtered = ' '.join(w for w in words if w not in LATIN_STOPWORDS)

    if filtered.strip():
        wc = WordCloud(
            width=700, height=420,
            background_color='#0A2342',
            colormap=cmap,
            max_words=80,
            prefer_horizontal=0.85,
            min_font_size=10,
            max_font_size=100,
            stopwords=LATIN_STOPWORDS,
            contour_width=0
        ).generate(filtered)
        ax.imshow(wc, interpolation='bilinear')
    else:
        ax.text(0.5, 0.5, 'Not enough data', ha='center', va='center',
                color='white', fontsize=14, transform=ax.transAxes)
        ax.set_facecolor('#0A2342')

    ax.set_title(f'{sentiment} Reviews', color='white', fontsize=14, fontweight='bold', pad=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('wordclouds_by_sentiment.png', dpi=150, bbox_inches='tight', facecolor='#0A2342')
plt.show()
print("Word clouds saved.")

### 📝 Figure 2 — Top Bigrams & Trigrams
Most frequent 2-word and 3-word phrases across all reviews.

In [ ]:
from nltk.util import ngrams
from nltk.corpus import stopwords
import matplotlib
import matplotlib.pyplot as plt
import re
from collections import Counter

STOP = set(stopwords.words('english')) | set(stopwords.words('french')) | {
    'app','volz','this','that','with','very','have','been','more','also',
    'good','great','just','like','mais','pour','avec','dans','bien','plus',
    'tout','cest','quil','really','even','would','could','should'
}

def get_ngrams(text_series, n, top_k=12):
    all_ng = []
    for text in text_series.dropna():
        tokens = [w for w in re.findall(r'\b[a-z]{3,}\b', str(text).lower()) if w not in STOP]
        all_ng.extend(ngrams(tokens, n))
    return Counter(all_ng).most_common(top_k)

bigrams  = get_ngrams(df['clean_content'], 2)
trigrams = get_ngrams(df['clean_content'], 3)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.patch.set_facecolor('#F8FCFD')
fig.suptitle('Most Frequent N-grams', fontsize=18, fontweight='bold', color='#0A2342')

def plot_ngrams(ax, data, title, cmap_name):
    if not data:
        ax.text(0.5, 0.5, 'Not enough data', ha='center', va='center', fontsize=14)
        ax.set_title(title); return
    labels = [' '.join(ng) for ng, _ in data]
    counts = [c for _, c in data]
    cmap = matplotlib.colormaps.get_cmap(cmap_name)
    bar_colors = [cmap(0.3 + 0.5 * i / len(data)) for i in range(len(data))]
    bars = ax.barh(labels[::-1], counts[::-1], color=bar_colors, edgecolor='white', linewidth=0.5)
    for bar, count in zip(bars, counts[::-1]):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                str(count), va='center', fontsize=9, color='#0A2342', fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', color='#0A2342', pad=10)
    ax.set_facecolor('#F0F9FF')
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(axis='y', labelsize=10)
    ax.set_xlabel('Frequency', fontsize=10, color='#64748B')

plot_ngrams(axes[0], bigrams,  'Top 12 Bigrams',  'Blues_r')
plot_ngrams(axes[1], trigrams, 'Top 12 Trigrams', 'Greens_r')

plt.tight_layout()
plt.savefig('ngrams_analysis.png', dpi=150, bbox_inches='tight', facecolor='#F8FCFD')
plt.show()
print("N-gram chart saved.")

### 🌍 Figure 3 — Sentiment by Language & Country
Compares sentiment distribution across scraping sources.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

palette = {'Positive': '#02C39A', 'Neutral': '#F39C12', 'Negative': '#E74C3C'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#F8FCFD')
fig.suptitle('', fontsize=18, fontweight='bold', color='#0A2342')

for ax, col, title in zip(axes, ['source_lang','source_country'], ['','by Country']):
    if col not in df.columns:
        ax.text(0.5, 0.5, f'Column "{col}" not found', ha='center', va='center'); continue
    ct = (df.groupby([col, 'sentiment'])
            .size()
            .unstack(fill_value=0)
            .apply(lambda r: r / r.sum() * 100, axis=1))
    for sent in ['Positive','Neutral','Negative']:
        if sent not in ct.columns: ct[sent] = 0
    ct[['Positive','Neutral','Negative']].plot(
        kind='bar', ax=ax, stacked=True,
        color=[palette[s] for s in ['Positive','Neutral','Negative']],
        edgecolor='white', linewidth=0.5
    )
    ax.set_title(title, fontsize=13, fontweight='bold', color='#0A2342')
    ax.set_xlabel('')
    ax.set_ylabel('Percentage (%)', fontsize=10)
    ax.set_facecolor('#F0F9FF')
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(axis='x', rotation=0, labelsize=10)
    ax.legend(title='Sentiment', bbox_to_anchor=(1.01,1), loc='upper left')

plt.tight_layout()
plt.savefig('sentiment_by_source.png', dpi=150, bbox_inches='tight', facecolor='#F8FCFD')
plt.show()
print("Sentiment by source chart saved.")

### 📏 Figure 4 — Review Length & Sentiment Over Time
Do unhappy users write longer reviews? Plus a stacked area chart of monthly sentiment.

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

palette = {'Positive': '#02C39A', 'Neutral': '#F39C12', 'Negative': '#E74C3C'}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('#F8FCFD')
fig.suptitle('Review Length Analysis', fontsize=18, fontweight='bold', color='#0A2342')

for sent, color in palette.items():
    subset = df[df['sentiment'] == sent]['word_count'].dropna()
    subset = subset[subset < 100]
    if len(subset) > 0:
        axes[0].hist(subset, bins=30, alpha=0.55, label=sent, color=color, edgecolor='white')

axes[0].set_title('Word Count Distribution', fontsize=13, fontweight='bold', color='#0A2342')
axes[0].set_xlabel('Number of Words', fontsize=10)
axes[0].set_ylabel('Count', fontsize=10)
axes[0].legend(title='Sentiment')
axes[0].set_facecolor('#F0F9FF')
axes[0].spines[['top','right']].set_visible(False)

plot_df = df[df['word_count'] < 100].copy()
order   = ['Negative','Neutral','Positive']
sns.boxplot(data=plot_df, x='sentiment', y='word_count', order=order,
            hue='sentiment', palette=palette, ax=axes[1],
            linewidth=1.2, fliersize=3, legend=False)
axes[1].set_title('Word Count by Sentiment (Boxplot)', fontsize=13, fontweight='bold', color='#0A2342')
axes[1].set_xlabel('Sentiment', fontsize=10)
axes[1].set_ylabel('Number of Words', fontsize=10)
axes[1].set_facecolor('#F0F9FF')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('review_length_analysis.png', dpi=150, bbox_inches='tight', facecolor='#F8FCFD')
plt.show()

# Stacked sentiment over time
df['at']         = pd.to_datetime(df['at'], errors='coerce')
df['year_month'] = df['at'].dt.to_period('M').astype(str)
monthly = (df.groupby(['year_month','sentiment'])
             .size().unstack(fill_value=0).sort_index())
for s in ['Positive','Neutral','Negative']:
    if s not in monthly.columns: monthly[s] = 0

fig, ax = plt.subplots(figsize=(16, 5))
fig.patch.set_facecolor('#F8FCFD')
monthly[['Negative','Neutral','Positive']].plot(
    kind='area', stacked=True, ax=ax,
    color=['#E74C3C','#F39C12','#02C39A'],
    alpha=0.75, linewidth=0
)
ax.set_title('Sentiment Composition Over Time', fontsize=16, fontweight='bold', color='#0A2342')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Number of Reviews', fontsize=11)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.set_facecolor('#F0F9FF')
ax.spines[['top','right']].set_visible(False)
ax.legend(title='Sentiment', loc='upper left')
plt.tight_layout()
plt.savefig('sentiment_over_time_stacked.png', dpi=150, bbox_inches='tight', facecolor='#F8FCFD')
plt.show()
print("Review length + stacked sentiment charts saved.")

---
# 💡 Phase 4 — Business Insights & Recommendations



### 📉 Figure A — Rating Funnel & Top Keywords per Star

In [ ]:
# ── Shared setup for all business insight figures ────────────────────────────
import pandas as pd, re, textwrap, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from collections import Counter
from itertools import combinations
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

df = pd.read_csv('volz_preprocessed_reviews.csv')
df['at'] = pd.to_datetime(df['at'], errors='coerce')

# ── All sentiment subsets ─────────────────────────────────────────────────────
neg  = df[df['sentiment'] == 'Negative'].copy()
neu  = df[df['sentiment'] == 'Neutral'].copy()
pos  = df[df['sentiment'] == 'Positive'].copy()

print(f"Total reviews   : {len(df)}")
print(f"  ✅ Positive   : {len(pos)}  ({round(len(pos)/len(df)*100,1)}%)")
print(f"  😐 Neutral    : {len(neu)}  ({round(len(neu)/len(df)*100,1)}%)")
print(f"  ❌ Negative   : {len(neg)}  ({round(len(neg)/len(df)*100,1)}%)")
print(f"     └─ 1★ : {len(df[df['score']==1])}   2★ : {len(df[df['score']==2])}")

# ── Shared stopwords ──────────────────────────────────────────────────────────
STOP = set(stopwords.words('english')) | set(stopwords.words('french')) | {
    'app','volz','application','this','that','with','very','have','been',
    'more','also','just','like','mais','pour','avec','dans','bien','plus',
    'tout','cette','cest','tres','quil','quon','when','really','just','even',
    'also','would','could','should','does','dont','cant','will','still'
}

SENT_PAL = {'Positive': '#02C39A', 'Neutral': '#F39C12', 'Negative': '#E74C3C'}

# ── Shared helper functions ───────────────────────────────────────────────────
def top_words(text_series, n=10):
    words = []
    for txt in text_series.dropna():
        tokens = re.findall(r'\b[a-z]{4,}\b', str(txt).lower())
        words += [w for w in tokens if w not in STOP]
    return Counter(words).most_common(n)

def build_graph(text_series, top_n=30, min_weight=2):
    all_words, tokenized = [], []
    for txt in text_series.dropna():
        tokens = [w for w in re.findall(r'\b[a-z]{4,}\b', str(txt).lower()) if w not in STOP]
        tokenized.append(tokens); all_words += tokens
    vocab = {w for w,_ in Counter(all_words).most_common(top_n)}
    edges = Counter()
    for tokens in tokenized:
        for pair in combinations(sorted(set(tokens) & vocab), 2):
            edges[pair] += 1
    G = nx.Graph()
    for (w1,w2), wt in edges.items():
        if wt >= min_weight:
            G.add_edge(w1, w2, weight=wt)
    return G

def find_kwic(text_series, keyword, n=3):
    """Extract sentences containing keyword from any review series."""
    mask = text_series.str.contains(keyword, case=False, na=False)
    results = []
    for review in text_series[mask].dropna():
        for sent in re.split(r'[.!?\n]', str(review)):
            if keyword.lower() in sent.lower() and len(sent.strip()) > 15:
                results.append(sent.strip()[:110]); break
        if len(results) >= n: break
    return results

# ── Topic keywords (applied to ALL reviews) ───────────────────────────────────
TOPICS = {
    '🐛 Bugs & Crashes': ['crash','bug','error','freeze','stop','broken','force','close','fix','problem','restart'],
    '⚡ Performance'    : ['slow','lag','speed','loading','fast','heavy','battery','memory','wait','respond'],
    '🎨 UI / UX'        : ['design','interface','button','screen','dark','theme','layout','confusing','navigate','ugly','hard'],
    '💳 Payments'       : ['payment','price','subscription','free','feature','premium','money','pay','cost','plan','charge'],
    '📶 Connectivity'   : ['connect','network','wifi','internet','offline','sync','server','login','account','access'],
}
topic_names = list(TOPICS.keys())
for name, kws in TOPICS.items():
    df[name]  = df['clean_content'].apply(
        lambda t: len(set(re.findall(r'\b[a-z]{3,}\b', str(t).lower())) & set(kws)))
    neg[name] = neg['clean_content'].apply(
        lambda t: len(set(re.findall(r'\b[a-z]{3,}\b', str(t).lower())) & set(kws)))

df['dominant_topic']  = df[topic_names].idxmax(axis=1)
df.loc[df[topic_names].max(axis=1)  == 0, 'dominant_topic'] = 'Other'
neg['dominant_topic'] = neg[topic_names].idxmax(axis=1)
neg.loc[neg[topic_names].max(axis=1) == 0, 'dominant_topic'] = 'Other'

all_topic_counts = df['dominant_topic'].value_counts()
all_topic_counts = all_topic_counts[all_topic_counts.index != 'Other']
neg_topic_counts = neg['dominant_topic'].value_counts()
neg_topic_counts = neg_topic_counts[neg_topic_counts.index != 'Other']

print("\n✅ Setup complete — df, neg, pos, neu, STOP, helpers, TOPICS all ready.")

# ── Topic Overview Plot ───────────────────────────────────────────────────────
import seaborn as sns

colors_t = ['#E74C3C','#F39C12','#3498DB','#9B59B6','#E07B39']
topic_labels = [t.split(' ',1)[1] if ' ' in t else t for t in all_topic_counts.index]

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.patch.set_facecolor('#F8FCFD')
fig.suptitle('Topic Overview — All Reviews', fontsize=18,
             fontweight='bold', color='#0A2342', y=1.02)

# ── Bar 1 : Total reviews per topic ──────────────────────────────────────────
ax = axes[0]
bars = ax.barh(topic_labels[::-1], all_topic_counts.values[::-1],
               color=colors_t[:len(all_topic_counts)][::-1],
               edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, all_topic_counts.values[::-1]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10, fontweight='bold', color='#0A2342')
ax.set_title('Volume per Topic (All Reviews)', fontsize=12,
             fontweight='bold', color='#0A2342')
ax.set_xlabel('Number of Reviews', fontsize=10)
ax.set_facecolor('#F0F9FF')
ax.spines[['top','right']].set_visible(False)

# ── Bar 2 : % Negative per topic (pain index) ────────────────────────────────
ax2 = axes[1]
pivot = (df[df['dominant_topic'] != 'Other']
         .groupby(['dominant_topic','sentiment'])
         .size().unstack(fill_value=0))
for col in ['Positive','Neutral','Negative']:
    if col not in pivot: pivot[col] = 0
pivot = pivot.loc[all_topic_counts.index]
pct_neg = (pivot['Negative'] / pivot.sum(axis=1) * 100).round(1)
pct_pos = (pivot['Positive'] / pivot.sum(axis=1) * 100).round(1)

bar_colors = ['#E74C3C' if p >= 50 else '#E07B39' if p >= 30 else '#F39C12'
              for p in pct_neg.values]
bars2 = ax2.barh(topic_labels[::-1], pct_neg.values[::-1],
                 color=bar_colors[::-1], edgecolor='white', linewidth=0.8)
for bar, val in zip(bars2, pct_neg.values[::-1]):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val}%', va='center', fontsize=10, fontweight='bold', color='#E74C3C')
ax2.axvline(x=50, color='#0A2342', linewidth=1.2, linestyle='--', alpha=0.4)
ax2.set_title('Pain Index — % Negative per Topic', fontsize=12,
              fontweight='bold', color='#0A2342')
ax2.set_xlabel('% of Negative Reviews', fontsize=10)
ax2.set_xlim(0, 100)
ax2.set_facecolor('#FFF5F5')
ax2.spines[['top','right']].set_visible(False)

# ── Bar 3 : Grouped Positive vs Negative per topic ───────────────────────────
ax3 = axes[2]
x    = np.arange(len(all_topic_counts))
w    = 0.3
bars_n = ax3.bar(x - w/2, pct_neg.values,  width=w, color='#E74C3C',
                 label='% Negative', edgecolor='white')
bars_p = ax3.bar(x + w/2, pct_pos.values,  width=w, color='#02C39A',
                 label='% Positive', edgecolor='white')

for bar in list(bars_n) + list(bars_p):
    h = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2, h + 0.8,
             f'{h:.0f}%', ha='center', va='bottom', fontsize=8,
             fontweight='bold', color='#0A2342')

ax3.set_xticks(x)
ax3.set_xticklabels(topic_labels, fontsize=8, rotation=15, ha='right')
ax3.set_title('Negative vs Positive % per Topic', fontsize=12,
              fontweight='bold', color='#0A2342')
ax3.set_ylabel('Percentage (%)', fontsize=10)
ax3.set_ylim(0, 110)
ax3.set_facecolor('#F0F9FF')
ax3.spines[['top','right']].set_visible(False)
ax3.legend(fontsize=9, frameon=False)

plt.tight_layout()
plt.savefig('fig_setup_overview.png', dpi=150, bbox_inches='tight', facecolor='#F8FCFD')
plt.show()

In [ ]:
import pandas as pd, re, matplotlib.pyplot as plt, matplotlib.patches as mpatches
from collections import Counter
from nltk.corpus import stopwords

if 'df' not in dir() or 'sentiment' not in df.columns:
    df  = pd.read_csv('volz_preprocessed_reviews.csv')
    df['at'] = pd.to_datetime(df['at'], errors='coerce')

STOP_A = set(stopwords.words('english')) | set(stopwords.words('french')) | {
    'app','volz','application','this','that','with','very','have','been',
    'more','also','just','like','mais','pour','avec','dans','bien','plus',
    'tout','cette','cest','tres','quil','quon','when','good','great'
}

def top_words_n(subset, n=6):
    words = []
    for txt in subset.dropna():
        tokens = re.findall(r'\b[a-z]{4,}\b', str(txt).lower())
        words += [w for w in tokens if w not in STOP_A]
    return Counter(words).most_common(n)

stars      = [5, 4, 3, 2, 1]
pal        = ['#02C39A','#52B788','#F39C12','#E07B39','#E74C3C']
star_counts = df['score'].value_counts()
max_count   = max(star_counts.get(s,0) for s in stars)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#F8FCFD')
fig.suptitle('Rating Funnel  &  Top Keywords per Star', fontsize=17,
             fontweight='bold', color='#0A2342', y=1.01)

for i, (star, color) in enumerate(zip(stars, pal)):
    count = star_counts.get(star, 0)
    w     = count / max_count if max_count else 0
    ax1.barh(i, w, left=(1-w)/2, color=color, height=0.6, edgecolor='white', linewidth=1.5)
    ax1.text(0.5, i, f'{star} ★   {count:,} reviews',
             ha='center', va='center', fontsize=12, fontweight='bold', color='white',
             transform=ax1.get_yaxis_transform())
ax1.set_xlim(0,1); ax1.set_title('Where users drop off', fontsize=13, color='#0A2342')
ax1.axis('off')

ax2.set_facecolor('#F8FCFD'); ax2.set_xlim(0,5); ax2.set_ylim(-0.5,7.5); ax2.axis('off')
ax2.set_title('Top 6 keywords per rating', fontsize=13, color='#0A2342')

for j, (star, color) in enumerate(zip(stars, pal)):
    subset = df[df['score'] == star]['clean_content']
    kw     = top_words_n(subset, 6)
    cx     = j + 0.5
    ax2.add_patch(mpatches.FancyBboxPatch((j+0.1,6.8),0.8,0.55,
                  boxstyle='round,pad=0.05', color=color, zorder=3))
    ax2.text(cx, 7.1, f'{star} ★', ha='center', va='center',
             fontsize=13, fontweight='bold', color='white', zorder=4)
    for k, (word, cnt) in enumerate(kw):
        y = 6.0 - k*0.95
        ax2.text(cx, y,    word, ha='center', va='center', fontsize=10, color='#1E293B')
        ax2.text(cx, y-0.38, f'({cnt})', ha='center', va='center',
                 fontsize=8, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('rating_funnel.png', dpi=150, bbox_inches='tight', facecolor='#F8FCFD')
plt.show()
print("Rating funnel saved.")

### 🗂️ Figure B — Topic Discovery: All Reviews with Sentiment Breakdown

In [ ]:
# ── Figure B : Topic Discovery — All Reviews, Sentiment Breakdown ────────────
colors_t = ['#E74C3C','#F39C12','#3498DB','#9B59B6','#E07B39']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#F8FCFD')
fig.suptitle('Topic Discovery — All Reviews with Sentiment Breakdown',
             fontsize=17, fontweight='bold', color='#0A2342', y=1.01)

# Donut — share of topics across ALL reviews
wedges, _, autotexts = axes[0].pie(
    all_topic_counts.values, autopct='%1.1f%%',
    colors=colors_t[:len(all_topic_counts)], startangle=140,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2), pctdistance=0.78)
for at in autotexts:
    at.set_fontsize(10); at.set_color('white'); at.set_fontweight('bold')
axes[0].legend(wedges, all_topic_counts.index, loc='lower center',
               bbox_to_anchor=(0.5,-0.12), fontsize=10, frameon=False)
axes[0].set_title('Topic Share (All Reviews)', fontsize=13, color='#0A2342')

# Stacked bar — topic × sentiment
pivot = (df[df['dominant_topic'] != 'Other']
         .groupby(['dominant_topic','sentiment'])
         .size().unstack(fill_value=0))
for col in ['Positive','Neutral','Negative']:
    if col not in pivot: pivot[col] = 0
pivot = pivot.loc[all_topic_counts.index]

x      = np.arange(len(pivot))
bottom = np.zeros(len(pivot))
for sent in ['Negative','Neutral','Positive']:
    vals = pivot[sent].values
    axes[1].bar(x, vals, bottom=bottom, color=SENT_PAL[sent],
                label=sent, edgecolor='white', linewidth=0.8)
    bottom += vals

axes[1].set_xticks(x)
axes[1].set_xticklabels([t.split(' ',1)[1] if ' ' in t else t
                         for t in pivot.index], fontsize=9, rotation=15, ha='right')
axes[1].set_title('Topic × Sentiment (All Reviews)', fontsize=13, color='#0A2342')
axes[1].set_ylabel('Number of Reviews', fontsize=10)
axes[1].set_facecolor('#F0F9FF')
axes[1].spines[['top','right']].set_visible(False)
axes[1].legend(title='Sentiment', loc='upper right', frameon=False)

plt.tight_layout()
plt.savefig('fig_B_topic_discovery.png', dpi=150, bbox_inches='tight', facecolor='#F8FCFD')
plt.show()

print('💡 TOPIC SUMMARY (All reviews):')
for topic in all_topic_counts.index:
    total = all_topic_counts[topic]
    n_neg = pivot.loc[topic,'Negative'] if 'Negative' in pivot.columns else 0
    n_pos = pivot.loc[topic,'Positive'] if 'Positive' in pivot.columns else 0
    print(f'  {topic}: {total} reviews  |  {round(n_neg/total*100)}% negative  |  {round(n_pos/total*100)}% positive')


### 🕸️ Figure C — Co-occurrence Network

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.patch.set_facecolor('#0A2342')
fig.suptitle('Co-occurrence Network — What Words Cluster Together?',
             fontsize=17, fontweight='bold', color='white', y=1.01)

subsets = [
    (neg['clean_content'], '#E74C3C', 'Negative Reviews — Pain Clusters'),
    (pos['clean_content'], '#02C39A', 'Positive Reviews — Strength Clusters'),
]

for ax, (text_series, color, title) in zip(axes, subsets):
    G = build_graph(text_series, top_n=30, min_weight=2)
    if len(G.nodes) == 0:
        ax.text(0.5,0.5,'Not enough data', ha='center', va='center',
                color='white', fontsize=14, transform=ax.transAxes)
        ax.set_facecolor('#0A2342'); ax.axis('off'); continue

    pos_layout = nx.spring_layout(G, seed=42, k=1.8)
    degrees    = dict(G.degree())
    weights    = [G[u][v]['weight'] for u,v in G.edges()]
    max_w      = max(weights) if weights else 1

    nx.draw_networkx_edges(G, pos_layout, ax=ax,
        width=[0.5+3*(w/max_w) for w in weights], edge_color=color, alpha=0.35)
    nx.draw_networkx_nodes(G, pos_layout, ax=ax,
        node_size=[250+degrees[n]*100 for n in G.nodes()], node_color=color, alpha=0.85)
    nx.draw_networkx_labels(G, pos_layout, ax=ax,
        font_size=8, font_color='white', font_weight='bold')

    central = sorted(nx.degree_centrality(G).items(), key=lambda x: -x[1])[:3]
    c_nodes = [n for n,_ in central]
    nx.draw_networkx_nodes(G, pos_layout, nodelist=c_nodes, ax=ax,
        node_size=[600+degrees[n]*120 for n in c_nodes], node_color='white', alpha=0.95)
    nx.draw_networkx_labels(G, pos_layout, labels={n:n for n in c_nodes}, ax=ax,
        font_size=9, font_color=color, font_weight='bold')

    ax.set_title(title, fontsize=14, fontweight='bold', color='white', pad=10)
    ax.set_facecolor('#0A2342'); ax.axis('off')

plt.tight_layout()
plt.savefig('fig_C_cooccurrence_network.png', dpi=150, bbox_inches='tight', facecolor='#0A2342')
plt.show()

---
#  Phase 5 — Enhancement 1: Arabic NLP Pipeline (CAMeL Tools)

**Motivation:** Volz raised $5M and is expanding into North & West Africa. The current pipeline discards all Arabic text via `[a-z]{4,}` regex. This cell processes Arabic reviews using CAMeL Tools and generates an Arabic word cloud.


In [ ]:
!pip install camel-tools --quiet
print("CAMeL Tools installed.")

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

try:
    from camel_tools.tokenizers.word import simple_word_tokenize
    CAMEL_AVAILABLE = True
except ImportError:
    CAMEL_AVAILABLE = False
    print("CAMeL Tools not available — using basic Arabic tokenizer as fallback.")

df = pd.read_csv('volz_preprocessed_reviews.csv')

# ── Arabic stopwords (MSA + Algerian Darija common words) ────────────────────
ARABIC_STOPS = {
    'في','من','على','إلى','عن','مع','هذا','هذه','التطبيق','تطبيق',
    'لا','كان','هو','هي','كل','أن','لكن','هناك','أو','وهو',
    'فولز','ولز','برنامج','هذا','ذلك','التي','الذي','وهي',
    'وقد','قد','أنا','نحن','كما','حتى','إذا','ما','كيف',
    'يمكن','يجب','عند','بعد','قبل','مثل','لكن','ولكن'
}

def detect_script(text):
    arabic_chars = len(re.findall(r'[\u0600-\u06FF]', str(text)))
    total_chars  = max(len(str(text)), 1)
    return 'arabic' if arabic_chars / total_chars > 0.3 else 'latin'

def tokenize_arabic(text):
    arabic_only = re.sub(r'[^\u0600-\u06FF\s]', ' ', str(text))
    if CAMEL_AVAILABLE:
        tokens = simple_word_tokenize(arabic_only)
    else:
        tokens = arabic_only.split()
    return [t for t in tokens if t not in ARABIC_STOPS and len(t) > 2]

df['script']         = df['content'].apply(detect_script)
arabic_df            = df[df['script'] == 'arabic'].copy()
arabic_df['ar_tokens'] = arabic_df['content'].apply(tokenize_arabic)

print(f"Total reviews       : {len(df)}")
print(f"Arabic-script reviews: {len(arabic_df)} ({round(len(arabic_df)/len(df)*100,1)}%)")
print(f"Latin-script reviews : {len(df) - len(arabic_df)}")

# ── Top Arabic words ──────────────────────────────────────────────────────────
all_ar_tokens = [t for tokens in arabic_df['ar_tokens'].dropna() for t in tokens]
top_arabic    = Counter(all_ar_tokens).most_common(20)
print("\nTop 20 Arabic keywords:")
for word, count in top_arabic:
    print(f"  {word}: {count}")

# ── Arabic word cloud per sentiment ───────────────────────────────────────────
try:
    from wordcloud import WordCloud
    FONT_PATH = None  # Set to path of an Arabic font if available e.g. 'NotoNaskhArabic-Regular.ttf'

    sentiments = ['Positive', 'Neutral', 'Negative']
    cmaps      = ['Greens',   'Oranges',  'Reds'   ]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.patch.set_facecolor('#0A2342')
    fig.suptitle('Arabic Word Clouds by Sentiment', fontsize=18,
                 fontweight='bold', color='white', y=1.02)

    for ax, sentiment, cmap in zip(axes, sentiments, cmaps):
        ar_sub  = arabic_df[arabic_df['sentiment'] == sentiment]['ar_tokens'].dropna()
        ar_text = ' '.join([t for tokens in ar_sub for t in tokens])

        if ar_text.strip() and len(ar_text.split()) >= 5:
            wc_kwargs = dict(
                width=700, height=420,
                background_color='#0A2342',
                colormap=cmap,
                max_words=60,
                min_font_size=12,
                max_font_size=90,
            )
            if FONT_PATH:
                wc_kwargs['font_path'] = FONT_PATH
            wc = WordCloud(**wc_kwargs).generate(ar_text)
            ax.imshow(wc, interpolation='bilinear')
        else:
            ax.text(0.5, 0.5, f'Not enough Arabic\ndata for {sentiment}',
                    ha='center', va='center', color='white', fontsize=12,
                    transform=ax.transAxes)
            ax.set_facecolor('#0A2342')

        ax.set_title(f'{sentiment} (Arabic)', color='white',
                     fontsize=13, fontweight='bold', pad=10)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('arabic_wordclouds.png', dpi=150, bbox_inches='tight', facecolor='#0A2342')
    plt.show()
    print("Arabic word clouds saved.")

except Exception as e:
    print(f"Word cloud error: {e}. Install a compatible Arabic font for best results.")

# ── Arabic sentiment breakdown ────────────────────────────────────────────────
ar_sent = arabic_df['sentiment'].value_counts(normalize=True).mul(100).round(1)
print("\nArabic reviews sentiment breakdown:")
print(ar_sent.to_string())

---
#  Phase 6 — Enhancement 2: App Version Sentiment Tracker

**Motivation:** The `appVersion` column exists in the data but was never used. Volz experienced a real rating crash in June 2025. This cell identifies exactly which app versions caused it — giving the team a release-level quality gate.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df = pd.read_csv('volz_preprocessed_reviews.csv')
df['at'] = pd.to_datetime(df['at'], errors='coerce')

df['appVersion'] = df['appVersion'].fillna('Unknown').astype(str).str.strip()

# Only analyze versions with ≥ 5 reviews
version_counts = df['appVersion'].value_counts()
valid_versions = version_counts[version_counts >= 5].index
df_v           = df[df['appVersion'].isin(valid_versions)].copy()

print(f"Total versions found  : {df['appVersion'].nunique()}")
print(f"Versions with ≥5 reviews: {len(valid_versions)}")

# ── Per-version stats ─────────────────────────────────────────────────────────
version_stats = df_v.groupby('appVersion').agg(
    avg_rating    = ('score', 'mean'),
    review_count  = ('score', 'count'),
    pct_negative  = ('sentiment', lambda x: round((x=='Negative').mean()*100, 1)),
    pct_positive  = ('sentiment', lambda x: round((x=='Positive').mean()*100, 1)),
).round(2).sort_values('avg_rating')

# Flag versions: avg rating < 3.0 OR negative rate > 40%
version_stats['ALERT'] = (
    (version_stats['avg_rating']   < 3.0) |
    (version_stats['pct_negative'] > 40.0)
)

print("\n⚠️  Versions needing attention:")
alerts = version_stats[version_stats['ALERT']]
if len(alerts):
    print(alerts[['avg_rating','pct_negative','review_count']].to_string())
else:
    print("  None — all versions above threshold.")

# ── Visualization ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, max(5, len(version_stats)*0.4 + 2)))
fig.patch.set_facecolor('#F8FCFD')
fig.suptitle('App Version — Sentiment & Rating Quality Gate',
             fontsize=16, fontweight='bold', color='#0A2342')

# Left: avg rating per version
bar_colors = ['#E74C3C' if a else '#02C39A' for a in version_stats['ALERT']]
bars = axes[0].barh(version_stats.index, version_stats['avg_rating'],
                    color=bar_colors, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, version_stats['avg_rating']):
    axes[0].text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
                 f'{val:.2f}', va='center', fontsize=9, color='#0A2342', fontweight='bold')
axes[0].axvline(x=3.0, color='#E74C3C', linestyle='--', linewidth=1.5,
                label='Alert threshold 3.0', alpha=0.7)
axes[0].set_title('Average Rating by Version', fontsize=13, fontweight='bold', color='#0A2342')
axes[0].set_xlabel('Average Star Rating', fontsize=10)
axes[0].set_xlim(0, 5.5)
axes[0].set_facecolor('#F0F9FF'); axes[0].spines[['top','right']].set_visible(False)
axes[0].legend(fontsize=9, frameon=False)

# Right: % negative per version
neg_colors = ['#E74C3C' if p>40 else '#F39C12' if p>25 else '#02C39A'
              for p in version_stats['pct_negative']]
bars2 = axes[1].barh(version_stats.index, version_stats['pct_negative'],
                     color=neg_colors, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars2, version_stats['pct_negative']):
    axes[1].text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                 f'{val}%', va='center', fontsize=9, color='#0A2342', fontweight='bold')
axes[1].axvline(x=40, color='#E74C3C', linestyle='--', linewidth=1.5,
                label='Alert threshold 40%', alpha=0.7)
axes[1].set_title('% Negative Reviews by Version', fontsize=13, fontweight='bold', color='#0A2342')
axes[1].set_xlabel('% Negative Reviews', fontsize=10)
axes[1].set_xlim(0, 105)
axes[1].set_facecolor('#FFF5F5'); axes[1].spines[['top','right']].set_visible(False)
axes[1].legend(fontsize=9, frameon=False)

# Red alert banner for flagged versions
if len(alerts):
    fig.text(0.5, -0.04,
             f"⚠️  {len(alerts)} version(s) flagged: {', '.join(alerts.index.tolist())}",
             ha='center', fontsize=11, color='#E74C3C', fontweight='bold')

plt.tight_layout()
plt.savefig('version_quality_gate.png', dpi=150, bbox_inches='tight', facecolor='#F8FCFD')
plt.show()
print("Version quality gate chart saved.")

---
#  Phase 7 — Enhancement 3: Payment Method Granularity

**Motivation:** 57% of Algerians are unbanked. Volz supports CIB card, Edahabia, and cash-on-delivery — three different user segments. The current code treats them as one "Payments" topic. This cell reveals which specific payment method drives the 60% pain index.


In [ ]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

df = pd.read_csv('volz_preprocessed_reviews.csv')

# ── Payment method keyword sets ───────────────────────────────────────────────
PAYMENT_METHODS = {
    'CIB Card'          : ['cib', 'carte', 'card', 'bank', 'bancaire', 'debit'],
    'Edahabia'          : ['edahabia', 'poste', 'ccp', 'postal', 'algerie poste'],
    'Cash-on-Delivery'  : ['livraison', 'domicile', 'especes', 'cash',
                           'delivery', 'livreur', 'comptant'],
    'Online / Other'    : ['paiement', 'payment', 'online', 'virement',
                           'transaction', 'payé', 'paid', 'prix', 'price',
                           'cher', 'cout', 'coût', 'tarif', 'charge'],
}

for method, kws in PAYMENT_METHODS.items():
    df[f'pay_{method}'] = df['content'].apply(
        lambda t: any(k in str(t).lower() for k in kws)
    )

# Reviews mentioning any payment method
pay_cols    = [f'pay_{m}' for m in PAYMENT_METHODS]
payment_df  = df[df[pay_cols].any(axis=1)].copy()
print(f"Reviews mentioning payment methods: {len(payment_df)} / {len(df)}")

# ── Per-method stats ──────────────────────────────────────────────────────────
method_stats = {}
for method in PAYMENT_METHODS:
    col    = f'pay_{method}'
    subset = df[df[col] == True]
    if len(subset) >= 3:
        method_stats[method] = {
            'count'      : len(subset),
            'avg_rating' : round(subset['score'].mean(), 2),
            'pct_negative': round((subset['sentiment']=='Negative').mean()*100, 1),
            'pct_positive': round((subset['sentiment']=='Positive').mean()*100, 1),
        }

ms_df = pd.DataFrame(method_stats).T.sort_values('pct_negative', ascending=False)
print("\nPayment method breakdown:")
print(ms_df.to_string())

# ── Visualization ─────────────────────────────────────────────────────────────
if len(ms_df):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor('#F8FCFD')
    fig.suptitle('Payment Method Pain Analysis — Which Method Needs Fixing Most?',
                 fontsize=15, fontweight='bold', color='#0A2342')

    colors_pay = ['#E74C3C' if p>50 else '#F39C12' if p>30 else '#02C39A'
                  for p in ms_df['pct_negative']]
    bars = axes[0].bar(ms_df.index, ms_df['pct_negative'], color=colors_pay,
                       edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, ms_df['pct_negative']):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                     f'{val}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[0].axhline(50, color='#E74C3C', linestyle='--', alpha=0.5, label='50% threshold')
    axes[0].set_title('Pain Index (% Negative) by Payment Method',
                      fontsize=12, fontweight='bold', color='#0A2342')
    axes[0].set_ylabel('% Negative Reviews', fontsize=10)
    axes[0].set_ylim(0, 105); axes[0].legend(fontsize=9, frameon=False)
    axes[0].set_facecolor('#FFF5F5'); axes[0].spines[['top','right']].set_visible(False)
    axes[0].tick_params(axis='x', rotation=10)

    axes[1].bar(ms_df.index, ms_df['avg_rating'],
                color=['#9B59B6']*len(ms_df), edgecolor='white', linewidth=0.8)
    for i, (idx, row) in enumerate(ms_df.iterrows()):
        axes[1].text(i, row['avg_rating']+0.05, f"{row['avg_rating']:.2f}★",
                     ha='center', va='bottom', fontsize=10, fontweight='bold', color='#0A2342')
    axes[1].axhline(3.0, color='#E74C3C', linestyle='--', alpha=0.5, label='Threshold 3.0★')
    axes[1].set_title('Average Rating by Payment Method',
                      fontsize=12, fontweight='bold', color='#0A2342')
    axes[1].set_ylabel('Average Star Rating', fontsize=10)
    axes[1].set_ylim(0, 5.5); axes[1].legend(fontsize=9, frameon=False)
    axes[1].set_facecolor('#F0F9FF'); axes[1].spines[['top','right']].set_visible(False)
    axes[1].tick_params(axis='x', rotation=10)

    plt.tight_layout()
    plt.savefig('payment_method_analysis.png', dpi=150, bbox_inches='tight', facecolor='#F8FCFD')
    plt.show()
    print("Payment method chart saved.")
else:
    print("Not enough payment-related reviews to plot.")

---
#  Phase 8 — Enhancement 4: B2B / Corporate Signal Detector

**Motivation:** Volz signed a partnership with Turkish Airlines and announced a corporate travel product. Some current users already book for business. Detecting these signals now tells the product team what corporate users expect before the official B2B launch.


In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
from collections import Counter

df = pd.read_csv('volz_preprocessed_reviews.csv')

# ── B2B keyword set (French + English + Arabic corporate terms) ──────────────
B2B_KEYWORDS = [
    # French
    'entreprise', 'société', 'facture', 'reçu', 'professionnel',
    'groupe', 'équipe', 'employé', 'mission', 'remboursement',
    'note de frais', 'voyage professionnel', 'billet professionnel',
    # English
    'company', 'invoice', 'receipt', 'corporate', 'business travel',
    'team', 'employees', 'reimbursement', 'expense', 'work trip',
    # Arabic corporate terms
    'شركة', 'فاتورة', 'عمل', 'مؤسسة', 'موظف', 'مهمة',
]

df['is_b2b'] = df['content'].apply(
    lambda t: any(k in str(t).lower() for k in B2B_KEYWORDS)
)

b2b = df[df['is_b2b'] == True].copy()
b2c = df[df['is_b2b'] == False].copy()

print(f"B2B-signal reviews : {len(b2b)} ({round(len(b2b)/len(df)*100,1)}%)")
print(f"B2C reviews        : {len(b2c)}")

if len(b2b) >= 3:
    print("\n--- B2B vs B2C Satisfaction Comparison ---")
    for label, subset in [('B2B', b2b), ('B2C', b2c)]:
        pct_neg = round((subset['sentiment']=='Negative').mean()*100, 1)
        pct_pos = round((subset['sentiment']=='Positive').mean()*100, 1)
        avg_r   = round(subset['score'].mean(), 2)
        print(f"  {label}: avg={avg_r}★ | neg={pct_neg}% | pos={pct_pos}%")

    # ── Top B2B pain words ────────────────────────────────────────────────────
    STOP_B2B = {'app','volz','application','this','that','with','very','have',
                'been','more','also','just','like','mais','pour','avec','dans',
                'bien','plus','tout','cest','quil'}
    def top_n(series, n=8):
        words = []
        for txt in series.dropna():
            tokens = re.findall(r'\b[a-z]{4,}\b', str(txt).lower())
            words += [w for w in tokens if w not in STOP_B2B]
        return Counter(words).most_common(n)

    b2b_neg_words = top_n(b2b[b2b['sentiment']=='Negative']['clean_content'])
    print("\nTop B2B complaint keywords:")
    for w, c in b2b_neg_words: print(f"  {w}: {c}")

    # ── Visualization ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#F8FCFD')
    fig.suptitle('B2B vs B2C — Satisfaction Comparison',
                 fontsize=15, fontweight='bold', color='#0A2342')

    SENT_PAL2 = {'Positive':'#02C39A','Neutral':'#F39C12','Negative':'#E74C3C'}

    for ax, (label, subset) in zip(axes, [('B2B Users', b2b), ('B2C Users', b2c)]):
        sent_pct = subset['sentiment'].value_counts(normalize=True).mul(100)
        for s in ['Positive','Neutral','Negative']:
            if s not in sent_pct: sent_pct[s] = 0
        bars = ax.bar(['Positive','Neutral','Negative'],
                      [sent_pct.get(s,0) for s in ['Positive','Neutral','Negative']],
                      color=[SENT_PAL2[s] for s in ['Positive','Neutral','Negative']],
                      edgecolor='white', linewidth=0.8)
        for bar, s in zip(bars, ['Positive','Neutral','Negative']):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                    f"{sent_pct.get(s,0):.1f}%", ha='center', va='bottom',
                    fontsize=10, fontweight='bold', color='#0A2342')
        ax.set_title(f'{label} (n={len(subset)})', fontsize=12,
                     fontweight='bold', color='#0A2342')
        ax.set_ylabel('% Reviews', fontsize=10)
        ax.set_ylim(0, 105)
        ax.set_facecolor('#F0F9FF'); ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    plt.savefig('b2b_vs_b2c.png', dpi=150, bbox_inches='tight', facecolor='#F8FCFD')
    plt.show()
    print("B2B vs B2C chart saved.")
else:
    print(f"Only {len(b2b)} B2B reviews found — not enough for meaningful analysis.")
    print("Tip: expand B2B_KEYWORDS list or lower the minimum threshold.")

---
#  Phase 9 — Enhancement 5: Diaspora vs Local User Segmentation

**Motivation:** The `fr/fr` and `en/us` scraping configs were deliberately included to capture Algerian diaspora users in France and the US. These users compare Volz to Booking.com and Skyscanner. Their complaints are higher-standard UX critiques — a different persona from local Algerian users.


In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
from collections import Counter
from nltk.corpus import stopwords

df = pd.read_csv('volz_preprocessed_reviews.csv')

# ── Segment by scraping config ────────────────────────────────────────────────
DIASPORA_CONFIGS = [('fr','fr'), ('en','us')]
LOCAL_CONFIGS    = [('ar','dz'), ('fr','dz'), ('en','dz')]

df['user_segment'] = df.apply(
    lambda r: 'Diaspora'
              if (r.get('source_lang',''), r.get('source_country','')) in DIASPORA_CONFIGS
              else 'Local',
    axis=1
)

diaspora = df[df['user_segment'] == 'Diaspora']
local    = df[df['user_segment'] == 'Local']

print(f"Diaspora users: {len(diaspora)} ({round(len(diaspora)/len(df)*100,1)}%)")
print(f"Local users   : {len(local)} ({round(len(local)/len(df)*100,1)}%)")

STOP2 = set(stopwords.words('english')) | set(stopwords.words('french')) | {
    'app','volz','application','this','that','with','very','have','been',
    'more','also','just','like','mais','pour','avec','dans','bien','plus',
    'tout','cest','quil','really','even','would','could'
}

def top_n(series, n=10):
    words = []
    for txt in series.dropna():
        tokens = re.findall(r'\b[a-z]{4,}\b', str(txt).lower())
        words += [w for w in tokens if w not in STOP2]
    return Counter(words).most_common(n)

print("\nDiaspora top complaints:")
d_neg = diaspora[diaspora['sentiment']=='Negative']
for w, c in top_n(d_neg['clean_content']): print(f"  {w}: {c}")

print("\nLocal top complaints:")
l_neg = local[local['sentiment']=='Negative']
for w, c in top_n(l_neg['clean_content']): print(f"  {w}: {c}")

# ── Visualization ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#F8FCFD')
fig.suptitle('Diaspora vs Local Users — Sentiment & Pain Points',
             fontsize=15, fontweight='bold', color='#0A2342')

SENT_PAL3 = {'Positive':'#02C39A','Neutral':'#F39C12','Negative':'#E74C3C'}

for ax, (label, subset) in zip(axes[:2], [('Diaspora (FR/US)', diaspora), ('Local Algeria', local)]):
    sent_pct = subset['sentiment'].value_counts(normalize=True).mul(100)
    for s in ['Positive','Neutral','Negative']:
        if s not in sent_pct: sent_pct[s] = 0.0
    vals  = [sent_pct.get(s,0) for s in ['Positive','Neutral','Negative']]
    bars  = ax.bar(['Positive','Neutral','Negative'], vals,
                   color=[SENT_PAL3[s] for s in ['Positive','Neutral','Negative']],
                   edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(f'{label}\n(n={len(subset)}, avg={subset["score"].mean():.2f}★)',
                 fontsize=11, fontweight='bold', color='#0A2342')
    ax.set_ylim(0, 105); ax.set_ylabel('% Reviews', fontsize=10)
    ax.set_facecolor('#F0F9FF'); ax.spines[['top','right']].set_visible(False)

# Right: side-by-side top complaint words
ax3 = axes[2]; ax3.axis('off'); ax3.set_facecolor('#F8FCFD')
ax3.set_title('Top Complaint Keywords by Segment', fontsize=11, fontweight='bold', color='#0A2342')

d_top = top_n(d_neg['clean_content'], 8)
l_top = top_n(l_neg['clean_content'], 8)
max_c = max([c for _,c in d_top+l_top] + [1])

for i, ((dw,dc),(lw,lc)) in enumerate(zip(d_top[:8], l_top[:8])):
    y = 0.92 - i*0.11
    ax3.text(0.02, y, dw, ha='left', va='center', fontsize=9,
             color='#2E86AB', fontweight='bold', transform=ax3.transAxes)
    ax3.text(0.30, y, str(dc), ha='left', va='center', fontsize=8,
             color='#2E86AB', transform=ax3.transAxes)
    ax3.text(0.55, y, lw, ha='left', va='center', fontsize=9,
             color='#E07B39', fontweight='bold', transform=ax3.transAxes)
    ax3.text(0.83, y, str(lc), ha='left', va='center', fontsize=8,
             color='#E07B39', transform=ax3.transAxes)

ax3.text(0.02, 1.0, 'Diaspora', ha='left', fontsize=10, fontweight='bold',
         color='#2E86AB', transform=ax3.transAxes)
ax3.text(0.55, 1.0, 'Local',    ha='left', fontsize=10, fontweight='bold',
         color='#E07B39', transform=ax3.transAxes)

plt.tight_layout()
plt.savefig('diaspora_vs_local.png', dpi=150, bbox_inches='tight', facecolor='#F8FCFD')
plt.show()
print("Diaspora vs local chart saved.")